In [18]:
import os 
import warnings
from dotenv import load_dotenv
import requests
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
from langsmith import Client
import certifi
from langchain.agents import create_react_agent,AgentExecutor
from langchain_core.tools import tool



In [19]:
warnings.filterwarnings(
    "ignore",
    message=r".*The `dict` method is deprecated.*",
    category=DeprecationWarning,
)

In [20]:
# # Save as test.py and run: python test.py

# from openai import OpenAI
# api_key=os.getenv("AWS_BEARER_TOKEN_BEDROCK")

# client = OpenAI(
#     api_key=api_key,
#     base_url="https://bedrock-mantle.ap-south-1.api.aws/v1",
#     project="default",
# )

# response = client.chat.completions.create(
#     model="openai.gpt-oss-120b",
#     messages=[{"role": "user", "content": "What is Amazon Bedrock?"}],
# )
# print(response.choices[0].message.content)

In [21]:
load_dotenv()
os.environ["SSL_CERT_FILE"] = certifi.where()
# openai_api_key = os.getenv("OPENROUTER_API_KEY")
tavily_api_key=os.getenv("TAVILY_API_KEY")



In [22]:
search_tool = TavilySearchResults(max_results=3, tavily_api_key=tavily_api_key)

In [23]:

@tool
def get_weather(city: str) -> dict:
    """Get the current weather information for a given city.
    
    Use this tool when the user asks about current weather,
    temperature, humidity, or weather conditions in a city.
    
    """
    url = f"https://api.weatherstack.com/current?access_key={os.getenv("WEATHER_STACK_API_KEY")}&query={city}"

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()

    if "error" in data:
        raise Exception(data["error"]["info"])

    return data

In [24]:
search_tool.invoke("What is the capital of France?")

[{'url': 'https://www.britannica.com/place/France',
  'content': 'The capital of France is Paris. Situated in the north-central part of the country, Paris is France\'s center of commerce and culture.\n\nFor centuries, Paris has been one of the world’s most attractive cities. Nicknamed the "City of Light" during the Enlightenment, Paris is known for business, commerce, study, culture, and entertainment. The city is appreciated for its gastronomy, haute couture, painting, literature, and intellectual community. Paris is home to the Eiffel Tower, one of the world\'s top tourist attractions.\n\nFrance has played a central role in European culture for much of its history. French artistic, culinary, and sartorial styles have influenced cultures worldwide, and remain a point of national pride. [...] The capital and by far the most important city of France is Paris, one of the world’s preeminent cultural and commercial centres. A majestic city known as the ville lumière, or “city of light,” Pa

In [25]:
llm=ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model="qwen/qwen3.8-27b",
    temperature=0.1
)

In [26]:
llm.invoke("Who won ipl 2026")

AIMessage(content='The **IPL 2026** season has not happened yet.\n\nAs of now, the most recent completed season is **IPL 2024**, which was won by the **Kolkata Knight Riders (KKR)**.\n\nThe IPL 2026 tournament is expected to take place in **April–May 2026**, so the winner will not be known until after the final match of that season.', response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 21, 'total_tokens': 113, 'completion_time': 0.17595664, 'completion_tokens_details': None, 'prompt_time': 0.001216683, 'prompt_tokens_details': None, 'queue_time': 0.052252016, 'total_time': 0.177173323}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'finish_reason': 'stop', 'logprobs': None}, id='run-6ab04bb7-f82c-43ba-a541-99d1e60b4c26-0')

In [27]:

client = Client()

prompt = client.pull_prompt("hwchase17/react")

print(prompt)

d:\Agentic_AI_Mastery\AI_AGENT\ai_agent_env\Lib\site-packages\langsmith\client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


In [28]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [29]:
tools=[search_tool,get_weather]


In [30]:
from datetime import date
from pydantic import BaseModel, Field


class ModelInfo(BaseModel):
    model_name: str = Field(description="Name of the AI model")
    release_date: date = Field(description="Release date of the AI model")
    company: str = Field(description="Company that released the model")
    message: str = Field(description="Short explanation about the model")

In [31]:
structured_llm = llm.with_structured_output(ModelInfo)

d:\Agentic_AI_Mastery\AI_AGENT\ai_agent_env\Lib\site-packages\langchain_core\_api\beta_decorator.py:87: LangChainBetaWarning: The function `with_structured_output` is in beta. It is actively being worked on, so the API may change.
  warn_beta(


In [32]:
agent = create_react_agent(
    llm=llm, 
    tools=tools, 
    prompt=prompt
    )

In [33]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [34]:
response = agent_executor.invoke({
    "input": ("Capital of Andhra Pradesh and get the wheather of that capital")
})



> Entering new AgentExecutor chain...
Question: Capital of Andhra Pradesh and get the weather of that capital
Thought: First, I need to identify the capital of Andhra Pradesh. I will search for this information.
Action: tavily_search_results_json
Action Input: Capital of Andhra Pradesh[{'url': 'https://en.wikipedia.org/wiki/Andhra_Pradesh', 'content': 'as Andhra Pradesh. While initially Hyderabad served as the capital for both the states, in 2017, the Government of Andhra Pradesh began operating from its new capital, Amaravati. A proposal to establish three capitals–Amaravati as the legislative capital, Visakhapatnam as the executive capital, and Kurnool as the judicial capital, was proposed by the state government in 2020 and was struck down by the Andhra Pradesh High Court. [...] | Consolidation | 1 November 1956 (69 years ago) |\n| Formation | 1 October 1953 (72 years ago) |\n|  | |\n| Capital | Amaravati |\n| Largest city | Visakhapatnam |\n| Districts | 28 |\n| Government | |\n|

In [35]:
print(response["output"])

The capital of Andhra Pradesh is Amaravati. The current weather in Amaravati is as follows:
- **Temperature:** 28°C (Feels like 33°C)
- **Conditions:** Patchy rain nearby
- **Humidity:** 86%
- **Wind:** 13 km/h from NNW
- **Cloud Cover:** 100%
- **Visibility:** 10 km
- **Local Time:** 18:42 (2026-09-23)


In [37]:
# Get structured output from the free-text agent answer.
# (The ReAct agent loop itself must use the plain llm, since it needs to
# emit free-form Thought/Action text and relies on the `stop` kwarg, which
# a with_structured_output()-wrapped runnable does not accept.)




# structured_result = structured_llm.invoke(response["output"])
# structured_result